# Kaggle runner
Source chính thức viết ở VS Code và quản lý trên GitHub. Gắn competition và Dataset private chứa `store-sales-source.zip`. Mỗi source change cần cập nhật Dataset version. Notebook chỉ thực thi. Chưa có điểm thực nghiệm thật trong file này.

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile
import json
import importlib.util
archives = list(Path("/kaggle/input").rglob("store-sales-source.zip"))
assert len(archives) == 1, f"Cần đúng một source zip; tìm thấy {archives}"
PROJECT = Path("/kaggle/working/store-sales-code")
assert not PROJECT.exists(), "Source đã giải nén: restart session trước khi đổi phiên source"
PROJECT.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.infolist():
        destination = (PROJECT / item.filename).resolve()
        assert destination.is_relative_to(PROJECT.resolve()), "Unsafe zip path"
    archive.extractall(PROJECT)
print(json.loads((PROJECT / "SOURCE_PROVENANCE.json").read_text()))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(PROJECT)], check=True)

## Chọn lần chạy
Lần đầu dùng baseline validation. Với LightGBM/LSTM kiểm tra dependencies, dùng smoke config trước. Fold tuning: 2017-06-12, 2017-06-28, 2017-07-14. Holdout 2017-07-30 chỉ dùng sau khi khóa lựa chọn. Muốn final submission đặt CUTOFF=None, dùng config đã chọn. GPU bật trong Settings nếu chạy LSTM.

In [ ]:
MODEL = "baseline"  # baseline / lightgbm / lstm
CUTOFF = "2017-07-14"  # None => final train/refit và tạo submission
CONFIG = "default.json"  # smoke.json chỉ kiểm tra luồng
RUN_ID = "baseline_fold3_v1"  # luôn đổi, không ghi đè
ALLOW_HOLDOUT = False
for dependency in {"baseline": [], "lightgbm": ["lightgbm"], "lstm": ["torch"]}[MODEL]:
    assert importlib.util.find_spec(dependency), f"Thiếu {dependency}; cài bản phù hợp và lưu version"
candidates = [p.parent for p in Path("/kaggle/input").rglob("train.csv") if (p.parent / "test.csv").exists() and (p.parent / "stores.csv").exists()]
assert len(candidates) == 1, f"Chọn DATA rõ ràng; tìm thấy {candidates}"
DATA = candidates[0]
OUT = Path("/kaggle/working") / RUN_ID
print(MODEL, CUTOFF, DATA, OUT)

In [ ]:
command = [sys.executable, "-m", "store_sales.run", "--data", str(DATA), "--config", str(PROJECT / "configs" / CONFIG), "--model", MODEL, "--out", str(OUT)]
if CUTOFF is not None:
    command += ["--cutoff", CUTOFF]
if ALLOW_HOLDOUT:
    command += ["--allow-holdout"]
subprocess.run(command, cwd=PROJECT, check=True)
print((OUT / "run.json").read_text())

## XAI khi đã chọn mô hình
Chỉ chạy trên run validation đã lưu. LightGBM cần SHAP/matplotlib. LSTM dùng conditional IG trong torch. Nếu run không phải mô hình tốt nhất, không trình bày nó như XAI mô hình thắng.

In [ ]:
RUN_XAI = False
if RUN_XAI:
    subprocess.run([sys.executable, "-m", "store_sales.explain", "--data", str(DATA), "--run", str(OUT)], cwd=PROJECT, check=True)

## Lưu phiên và submit
Save Version → Save & Run All. Với final mode, kiểm tra `/kaggle/working/<RUN_ID>/submission.csv` trong Outputs. Nộp đúng file đó; ghi notebook version, source SHA/hash, run ID và Kaggle score vào experiment log. Giữ checkpoint và environment. Không submit nhãn giả hay dùng bản smoke như kết quả cuối.